# Re-evaluate saved probes

Loads the probe weights a run wrote and scores them again on the test set, so a
comparison table can be rebuilt without re-running any sampling.

This is also where **PALM** and every other curve-level metric are fitted. A
run itself reports only accuracy, precision, recall and macro-F1: a curve fit
needs the whole budget sweep to have finished, which a resumed or GPU-sharded
run cannot guarantee while it is still going, and re-fitting here costs seconds
against re-running the sweep.

Attach the checkpoint dataset produced by `run_al_baseline.ipynb` (baselines)
or `run_al_sampler.ipynb` (scalpel).

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import numpy as np
import torch

from data.loaders import get_data_loaders
from evaluation.metrics import evaluate_probe
from evaluation.palm import format_palm_report, palm_evaluate
from features.visual import DINOv2Extractor, extract_image_features
from training.checkpoint import load_probe
from utils import set_seed

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"
SEED = 42

# One name per run. main.py derives them from the sampler config: `scalpel`
# becomes scalpel_<uncertainty_mode>[_<cell_pooling>][_<missing_impute>], every
# other sampler just uses its own name.
RUN_NAMES = [
    "scalpel_disagreement",
    "scalpel_visual_margin",
    "uncertainty_herding",
]

CHECKPOINT_ROOT = "/kaggle/input/EDIT_RUN_OUTPUT_SLUG/checkpoints"

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

device = torch.device(config["device"])
data_path = Path(DATA_PATHS[DATASET])
checkpoint_dir = Path(CHECKPOINT_ROOT) / DATASET
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert checkpoint_dir.is_dir(), f"Missing checkpoint directory: {checkpoint_dir}"

set_seed(SEED)
_, test_loader, _ = get_data_loaders(str(data_path), SEED, verbose=True)
test_dataset = test_loader.dataset
test_labels = (
    test_dataset.lbl
    if hasattr(test_dataset, "lbl")
    else np.array(test_dataset.dataset.targets)[test_dataset.indices]
)

extractor = DINOv2Extractor(
    model_name=config.get("models", {}).get("vit", "facebook/dinov2-base")
).to(device)
test_features = extract_image_features(test_loader, extractor, device)
del extractor
print("test features:", test_features.shape)

In [ ]:
budgets = config["cumulative_budget"]
accuracy = {}

for run_name in RUN_NAMES:
    accuracy[run_name] = {}
    for budget in budgets:
        checkpoint = checkpoint_dir / f"{run_name}_probe_budget_{budget}.pt"
        assert checkpoint.is_file(), f"Missing checkpoint: {checkpoint}"
        probe = load_probe(str(checkpoint), device)
        accuracy[run_name][budget] = evaluate_probe(
            probe, test_features, test_labels, device, verbose=False
        )[0]
        del probe

header = "budget".ljust(9) + "".join(f"{name:>26}" for name in RUN_NAMES)
print(header)
for budget in budgets:
    row = "".join(f"{accuracy[name][budget]:>26.4f}" for name in RUN_NAMES)
    print(f"{budget:<9}{row}")

In [ ]:
for run_name in RUN_NAMES:
    curve = accuracy[run_name]
    if len(curve) < 4:
        print(f"[PALM] {run_name}: need >= 4 budgets, got {len(curve)}")
        continue
    try:
        params = palm_evaluate(budgets=list(curve), accuracies=list(curve.values()))
    except (RuntimeError, ValueError) as error:
        print(f"[PALM] {run_name}: fitting failed: {error}")
        continue
    print(format_palm_report(params, run_name, DATASET))